In [ ]:
# =============================================================================
# jobs/26_warning_fgs_coverage.py
# Was each flood alert and warning under an FGS risk polygon, and was that forecast
# issued before it, on the day, or after?
#
# One row per warning episode (job 25), carrying how it was covered. The app's Warnings vs
# FGS tab reads it, to look at one warning and to measure how often the polygons and the
# warnings agree.
#
# What this script does:
#   1. Reads the links between FGS polygons and EA areas (fgs_ea_area_intersections, job 07)
#      for every complete statement: which areas a polygon was linked to, for which forecast
#      date, at what risk, in a statement issued on which day. A coastal warning area linked
#      through its parent alert area is included, as job 07 wrote it.
#   2. Reads ea_warning_episodes (job 25) and the flood warning and alert area tables for names.
#   3. For each episode, finds the statements that covered its area for any date it was in
#      force (utils/fgs_coverage.py has the rules) and records the worst risk, the worst risk
#      from statements issued before, on and after its issue day, and the earliest statement
#      at each risk rank (first_cover_ge1 .. first_cover_ge4). The app turns that into
#      before, same day, after or not covered for the risk the user chooses.
#   4. Writes ea_warning_fgs_coverage in full.
#
# NOT A MISSED FORECAST. The FGS forecasts risk across regions; the EA issues warnings for
# specific rivers and coasts. A warning with no polygon over it is a disagreement between
# the two, not necessarily an error in either.
#
# Run after 25 (Master_polling keeps it current). Exit values: "success".
# =============================================================================

import os
import sys

# Resolve the project root from this notebook's own location, so nothing
# depends on one person's workspace folder.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, LongType, BooleanType, DoubleType

from config import (
    TBL_STATEMENTS, TBL_FGS_INTERSECT_LOG, TBL_FGS_EA_INTERSECT, TBL_WARNING_FGS_COVERAGE,
    TBL_WARNING_EPISODES, TBL_EA_FWA, TBL_EA_FAA, FGS_GEOMETRY_SOURCE
)
from utils.helpers import get_spark, utc_now, table_exists, source_predicate, risk_rank_sql
from utils.fgs_coverage import cover_episodes, category, RANKS


# =============================================================================
# SETUP
# =============================================================================

spark = get_spark()
now   = utc_now()

for table in (TBL_STATEMENTS, TBL_FGS_INTERSECT_LOG, TBL_FGS_EA_INTERSECT, TBL_WARNING_EPISODES):
    if not table_exists(spark, table):
        raise Exception(f"{table} does not exist. Run 01/02, 07 and 25 first.")

print(f"FGS_GEOMETRY_SOURCE = {FGS_GEOMETRY_SOURCE}")


In [ ]:
# =============================================================================
# THE COVERAGE: which areas were under which polygons, by forecast date and issue day
# Reduced in Spark to one row per area, forecast date and issue day, so little crosses to pandas.
# =============================================================================

l_in_source = source_predicate(spark, TBL_FGS_INTERSECT_LOG, FGS_GEOMETRY_SOURCE, alias="l")
i_in_source = source_predicate(spark, TBL_FGS_EA_INTERSECT, FGS_GEOMETRY_SOURCE, alias="i")

coverage = spark.sql(f"""
    WITH complete AS (
        SELECT s.statement_id, to_date(s.issued_at) AS s_day
        FROM {TBL_FGS_INTERSECT_LOG} l
        JOIN {TBL_STATEMENTS} s
          ON s.statement_id = l.statement_id
         AND COALESCE(s.last_modified_at, '') = COALESCE(l.last_modified_at, '')
        WHERE {l_in_source}
    )
    SELECT i.ea_area_code,
           to_date(i.forecast_date)                                  AS d,
           c.s_day                                                   AS s,
           MAX({risk_rank_sql('i.risk_level')})                      AS rk,
           concat_ws(',', sort_array(collect_set(i.source)))         AS sources
    FROM {TBL_FGS_EA_INTERSECT} i
    JOIN complete c ON c.statement_id = i.statement_id
    WHERE {i_in_source}
      AND i.forecast_date IS NOT NULL
      AND {risk_rank_sql('i.risk_level')} >= 1
      AND i.ea_area_code IN (SELECT DISTINCT ea_area_code FROM {TBL_WARNING_EPISODES})
    GROUP BY i.ea_area_code, to_date(i.forecast_date), c.s_day
""").toPandas()
print(f"Coverage rows (area, forecast date, issue day): {len(coverage):,}   "
      f"areas: {coverage['ea_area_code'].nunique():,}   "
      f"issue days: {coverage['s'].min()} to {coverage['s'].max()}")

# =============================================================================
# THE EPISODES, AND HOW EACH WAS COVERED
# =============================================================================

episodes = spark.table(TBL_WARNING_EPISODES).select(
    "episode_id", "ea_area_code", "area_type", "start_time", "end_time", "duration_minutes",
    "peak_level", "had_severe", "stale_open").toPandas()
names = spark.sql(f"""
    SELECT ea_area_code, MAX(ea_area_name) AS ea_area_name FROM (
        SELECT ea_area_code, ea_area_name FROM {TBL_EA_FWA}
        UNION ALL
        SELECT ea_area_code, ea_area_name FROM {TBL_EA_FAA}
    ) GROUP BY ea_area_code
""").toPandas()

covered = cover_episodes(episodes, coverage, now.replace(tzinfo=None))
result = (episodes.merge(covered.drop(columns=["ea_area_code", "area_type"]), on="episode_id", how="left")
                  .merge(names, on="ea_area_code", how="left"))
result["computed_at"] = now.isoformat()
print(f"Episodes: {len(result):,}   with any covering polygon: {int((result['max_rank'] > 0).sum()):,}")
if len(result) == 0:
    raise Exception("No episodes: run 25 first.")

# A quick read, for each risk threshold: how the warnings were covered.
print("\nEvery alert and warning, by how the FGS covered it at each risk level:")
rows = []
for r in RANKS:
    cats = pd.Series([category(i, f) for i, f in zip(result["issue_date"], result[f"first_cover_ge{r}"])]).value_counts()
    n = len(result)
    rows.append({"at_or_above": {1: "Very Low", 2: "Low", 3: "Medium", 4: "High"}[r],
                 **{c: f"{100 * cats.get(c, 0) / n:.0f}%" for c in ("before", "same_day", "after", "none")}})
print(pd.DataFrame(rows).to_string(index=False))


In [ ]:
# =============================================================================
# WRITE TO DELTA
# Full overwrite: cheap to rebuild, and an amended statement or a late warning message
# changes the past.
# =============================================================================

SCHEMA = StructType([
    StructField("episode_id",        StringType(),  True),
    StructField("ea_area_code",      StringType(),  True),
    StructField("ea_area_name",      StringType(),  True),
    StructField("area_type",         StringType(),  True),
    StructField("start_time",        StringType(),  True),
    StructField("end_time",          StringType(),  True),
    StructField("duration_minutes",  DoubleType(),  True),
    StructField("peak_level",        StringType(),  True),
    StructField("had_severe",        BooleanType(), True),
    StructField("stale_open",        BooleanType(), True),
    StructField("issue_date",        DateType(),    True),
    StructField("max_rank",          IntegerType(), True),
    StructField("rank_before",       IntegerType(), True),
    StructField("rank_same_day",     IntegerType(), True),
    StructField("rank_after",        IntegerType(), True),
    StructField("first_cover_ge1",   DateType(),    True),
    StructField("first_cover_ge2",   DateType(),    True),
    StructField("first_cover_ge3",   DateType(),    True),
    StructField("first_cover_ge4",   DateType(),    True),
    StructField("sources",           StringType(),  True),
    StructField("computed_at",       StringType(),  True),
])

out = result.copy()
for c in ("start_time", "end_time"):
    out[c] = pd.to_datetime(out[c]).dt.strftime("%Y-%m-%dT%H:%M:%S")
for c in ("issue_date", "first_cover_ge1", "first_cover_ge2", "first_cover_ge3", "first_cover_ge4"):
    out[c] = pd.to_datetime(out[c]).dt.date
for c in ("max_rank", "rank_before", "rank_same_day", "rank_after"):
    out[c] = out[c].fillna(0).astype("int32")
out["duration_minutes"] = pd.to_numeric(out["duration_minutes"], errors="coerce")
out["had_severe"] = out["had_severe"].fillna(False).astype(bool)
out["stale_open"] = out["stale_open"].fillna(False).astype(bool)
out = out[[f.name for f in SCHEMA.fields]].astype(object).where(out[[f.name for f in SCHEMA.fields]].notna(), None)

(
    spark.createDataFrame(out, SCHEMA).write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TBL_WARNING_FGS_COVERAGE)
)

spark.sql(f"""
    COMMENT ON TABLE {TBL_WARNING_FGS_COVERAGE} IS
    'Each flood alert and warning (an episode from ea_warning_episodes) and how the FGS covered
     it. An area is covered when a risk polygon was linked to it (fgs_ea_area_intersections) for a
     date the warning was in force. first_cover_geN is the day the earliest statement was issued
     whose polygon over the area was at risk rank N or higher (1 Very Low .. 4 High); compare it with
     issue_date: earlier is before, equal is the same day, later is after, null is not covered.
     max_rank is the worst rank over any statement; rank_before, rank_same_day and rank_after are the
     worst from statements issued before, on and after issue_date. A warning with no polygon over it
     is a disagreement, not necessarily a missed forecast. Rebuilt daily by 26_warning_fgs_coverage.'
""")

written = spark.table(TBL_WARNING_FGS_COVERAGE).count()
print(f"\nVerified: {written:,} rows written to {TBL_WARNING_FGS_COVERAGE}.")

dbutils.notebook.exit("success")
